In [1]:
# 1. Initialization

import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets
from torchvision.transforms import v2
import numpy as np
import torch
from torch import nn

def get_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda:0")
    else:
        return torch.device("cpu")

device = get_device()

print('torch:', torch.__version__)
print('built CUDA:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print(f"Selected device: {device}")

torch: 2.13.0+cu130
built CUDA: 13.0
CUDA available: True
device: NVIDIA GeForce RTX 5060 Laptop GPU
Selected device: cuda:0


In [2]:
# Review

class ReviewRegressor(nn.Module):
    def __init__(
        self,
        num_features: int,
    )->None:
        super().__init__()
        self.linear = nn.Linear(
            in_features= num_features,
            out_features= 1,
        )

    def forward(
        self,
        x: torch.Tensor
    ):
        output = self.linear(x)
        return output.squeeze(-1)

num_features = 3

review_model = ReviewRegressor(num_features=num_features).to(device)

review_x = torch.randn(
    8,
    3,
    device=device
)

review_output = review_model(review_x)

assert review_output.shape == (8,)
assert review_output.device == device
assert len(list(review_model.parameters())) == 2

In [3]:
review_loss = nn.MSELoss(
    reduction='mean'
)

review_optimizer = torch.optim.SGD(
    review_model.parameters(),
    lr=0.05
)

# Since we don't have y generated, I'm just putting this in comment

"""
int max_epoch = 300
for epoch in range(1, max_epoch + 1):
    review_optimizer.zero_grad()
    output = review_model(review_x)
    loss = review_loss(output,y)

    loss.backward()

    review_optimizer.step()
"""

'\nint max_epoch = 300\nfor epoch in range(1, max_epoch + 1):\n    review_optimizer.zero_grad()\n    output = review_model(review_x)\n    loss = review_loss(output,y)\n\n    loss.backward()\n\n    review_optimizer.step()\n'

In [8]:
# Loading dataset

training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

In [13]:
print(type(training_data))
print(len(training_data))

<class 'torchvision.datasets.mnist.FashionMNIST'>
60000


In [ ]:
# Creating data loader

batch_size = 128

train_loader = DataLoader(
    training_data,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0
)

In [ ]:
# Inspect one batch

images_cpu, labels_cpu = next(iter(train_loader))

print(f"images shape: {images_cpu.shape}")
print(f"images dtype: {images_cpu.dtype}")
print(f"images range: [{images_cpu.min()}, {images_cpu.max()}]")

print(f"labels shape: {labels_cpu.shape}")
print(f"labels dtype: {labels_cpu.dtype}")
print(f"labels range: [{labels_cpu.min()}, {labels_cpu.max()}]")


"""
    for images:
        - Dim 0: no of images
        - Dim 1: no of color channels (1 for grey scale)
        - Dim 2,3: height, width respectively.

    for labels:
        - Dim 0: no of labels, one of each image lol 
"""

images shape: torch.Size([128, 1, 28, 28])
images dtype: torch.float32
images range: [0.0, 1.0]
labels shape: torch.Size([128])
labels dtype: torch.int64
labels range: [0, 9]


In [20]:
class FashionMLP(nn.Module):
    def __init__(
        self,
        hidden_features: int = 128
    ) -> None:
        super().__init__()

        self.flatten = nn.Flatten(
            start_dim = 1,
        )

        self.network = nn.Sequential(
            nn.Linear(
                in_features = 28 * 28,
                out_features = hidden_features
            ),
            nn.ReLU(),
            nn.Linear(
                in_features = hidden_features,
                out_features = 10
            ),
        )

    def forward(
        self,
        x: torch.Tensor,
    ) -> torch.Tensor:
        x = self.flatten(x)
        logits = self.network(x)
        return logits

In [23]:
model = FashionMLP(
    hidden_features=128,
).to(device)

images = images_cpu.to(device)
labels = labels_cpu.to(device)

logits = model(images)

print(images.shape)         # 128, 1, 28, 28
print(labels.shape)         # 128
print(logits.shape)         # 128, 10

print(logits.requires_grad) # True

torch.Size([128, 1, 28, 28])
torch.Size([128])
torch.Size([128, 10])
True


In [ ]:
parameter_count = sum(
    parameter.numel()
    for parameter in model.parameters()
)

print(f"Trainable parameters: {parameter_count}")

# 101770
# = (784 * 128 + 128) + (128 * 10 + 10)

Trainable parameters: 101770


In [ ]:
# loss & optimizers

loss_fn = nn.CrossEntropyLoss(
    reduction="mean"
)

learning_rate = 0.1

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=learning_rate
)

initial_loss = loss_fn(
    logits,
    labels
)

print(initial_loss)

tensor(2.3292, device='cuda:0', grad_fn=<NllLossBackward0>)


In [ ]:
num_epochs = 50

epoch_metrics: list[dict[str, float]] = []

for epoch in range(num_epochs):
    # Setting model to train mode
    model.train()                   

    # Initializing validating metrics
    running_loss_sum = 0.0
    running_correct = 0
    examples_seen = 0

    # Iterating over batches
    for batch_index, (
        images_cpu,
        labels_cpu,
    ) in enumerate(train_loader):
        images = images_cpu.to(device)
        labels = labels_cpu.to(device)

        # Resetting gradient
        optimizer.zero_grad(
            set_to_none=True
        )

        # Standard forward pass
        logits = model(images)
        loss = loss_fn(logits, labels)

        # Backprop
        loss.backward()

        # Updating params
        optimizer.step()

        current_batch_size = images.shape[0]

        # Updating running validation metrics
        running_loss_sum += loss.item() * current_batch_size
        predictions = logits.argmax(dim=1)
        running_correct += (predictions == labels).sum().item()
        examples_seen += current_batch_size

        # Logging
        if batch_index % 100 == 0:
            running_mean_loss = running_loss_sum / examples_seen
            running_accuracy = running_correct / examples_seen

            print(
                f"epoch = {epoch + 1} "
                f"batch = {batch_index} "
                f"loss = {running_mean_loss:.4f}"
                f"accuracy = {running_accuracy:.4f}"
            )

    # Calculating mean loss & accuracy for logging purpose
    epoch_loss = running_loss_sum / examples_seen
    epoch_accuracy = running_correct / examples_seen

    epoch_metrics.append(
        {
            "loss": epoch_loss,
            "accuracy": epoch_accuracy
        }
    )

    print(
        f"epoch={epoch + 1} complete "
        f"loss={epoch_loss:.4f} "
        f"accuracy={epoch_accuracy:.2%}"
    )

epoch = 1 batch = 0 loss = 0.5578accuracy = 0.8281
epoch = 1 batch = 100 loss = 0.4513accuracy = 0.8410
epoch = 1 batch = 200 loss = 0.4471accuracy = 0.8433
epoch = 1 batch = 300 loss = 0.4438accuracy = 0.8441
epoch = 1 batch = 400 loss = 0.4434accuracy = 0.8437
epoch=1 complete loss=0.4385 accuracy=84.51%
epoch = 2 batch = 0 loss = 0.3488accuracy = 0.8516
epoch = 2 batch = 100 loss = 0.4087accuracy = 0.8517
epoch = 2 batch = 200 loss = 0.4068accuracy = 0.8545
epoch = 2 batch = 300 loss = 0.4085accuracy = 0.8545
epoch = 2 batch = 400 loss = 0.4091accuracy = 0.8534
epoch=2 complete loss=0.4072 accuracy=85.48%
epoch = 3 batch = 0 loss = 0.3768accuracy = 0.8438
epoch = 3 batch = 100 loss = 0.3945accuracy = 0.8626
epoch = 3 batch = 200 loss = 0.3909accuracy = 0.8630
epoch = 3 batch = 300 loss = 0.3885accuracy = 0.8628
epoch = 3 batch = 400 loss = 0.3867accuracy = 0.8633
epoch=3 complete loss=0.3870 accuracy=86.34%
epoch = 4 batch = 0 loss = 0.3658accuracy = 0.8672
epoch = 4 batch = 100 los

In [ ]:
assert len(epoch_metrics) == num_epochs

assert epoch_metrics[-1]["loss"] > 0.0
assert epoch_metrics[-1]["accuracy"] > 0.80

In [ ]:
# Print predictions from last batch

print("targets:    ", labels[:10])
print(
    "predictions:",
    logits.argmax(dim=1)[:10],
)

targets:     tensor([7, 4, 8, 1, 3, 1, 4, 3, 6, 8], device='cuda:0')
predictions: tensor([7, 4, 8, 1, 3, 1, 4, 3, 6, 8], device='cuda:0')


In [ ]:
# Print probability from last batch, first image

with torch.no_grad():
    first_probabilities = torch.softmax(
        logits[0],
        dim=0,
    )

    print(first_probabilities)

tensor([4.2915e-12, 6.7862e-10, 4.2719e-12, 7.9955e-07, 2.6618e-11, 9.8969e-06,
        1.0636e-10, 9.9999e-01, 9.3173e-10, 6.8282e-08], device='cuda:0')
